In [1]:
import numpy as np
import pandas as pd
from joblib import dump, load
import os
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error, 
    mean_squared_error,
    root_mean_squared_error, 
    mean_absolute_percentage_error,
    root_mean_squared_log_error,
    make_scorer
)
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

### Data importation

In [2]:
df = pd.read_csv("../data/processed/bc_clean.csv")
df = df.drop(["Unnamed: 0"], axis=1)
df.head(3)

,latitude,longitude,price,property-beds,property-baths,Acreage,Property Tax,Square Footage,Missing Acreage,Missing Property Tax,...,heat_pump,overhead,space_heater,Property Type_Condo,Property Type_Condo/Townhome,Property Type_Duplex,Property Type_Manufactured Home,Property Type_MultiFamily,Property Type_Single Family,Property Type_Townhome
0,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,6995.0,4374.0,0,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,49.138904,-122.654191,1399999.0,6.0,4.0,0.04,2585.0,2404.0,0,0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,49.103726,-122.663125,399900.0,1.0,1.0,0.00,1474.0,632.0,1,0,...,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
X = df.drop(["price"], axis=1)
Y = df["price"]

#### Let's transform the Y vector target to a log-Y target vector in order to reduce the errors for the smallest prices of our dataset.  

Indeed, by applying the logarithm to the `Price` feature, the model penalizes much more the errors for the smallest price values than the biggest ones.  
We do that in order to give more importance to the errors on the smallest prices, by removing the dominance of large prices. We change the "space" of the error space thanks to the form of the logartihm function that becomes flatter for large values.

In [4]:
Y_log = np.log1p(Y)

#### train test split

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y_log, test_size=0.20, random_state=42, shuffle=True
)

#### BayesSearchCV

In [18]:
gdb_search_space = {
    "n_estimators" : Integer(20, 600),
    "learning_rate" : Real(0.01, 0.1),
    # We reduce the amount of samples for the training of each tree in order to reduce overfitting.
    # We make sure that each tree will take a percentage of its dataset (less than 100%).
    "subsample" : Real(0.6, 0.9),
    
    "max_depth": Integer(3, 9),
    "min_samples_split" : Integer(30, 100),
    "min_samples_leaf" : Integer(20, 100),
    "max_features": Categorical([None, "sqrt", "log2"])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

In [20]:
gdb_search = BayesSearchCV(
    estimator = GradientBoostingRegressor(),
    search_spaces = gdb_search_space,
    scoring = scoring["neg_rmsle"],
    n_iter = 60,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [24]:
if os.path.isfile("../artifacts/gdb_search.pkl"):
    print("The object already exists")
else : 
    gdb_search.fit(X_train, Y_train)
    dump(gdb_search, "../artifacts/gdb_search.pkl")
    print("The object has been successfully saved")

The object already exists


In [25]:
gdb_search = load("../artifacts/gdb_search.pkl")
gdb_search.best_params_

OrderedDict([('learning_rate', 0.058275008984106036),
             ('max_depth', 9),
             ('max_features', None),
             ('min_samples_leaf', 20),
             ('min_samples_split', 30),
             ('n_estimators', 600),
             ('subsample', 0.6)])

In [26]:
best_model_gdb = gdb_search.best_estimator_

y_pred_log = best_model_gdb.predict(X_test)
y_test_pred = np.expm1(y_pred_log)

y_train_pred_log = best_model_gdb.predict(X_train)
y_train_pred = np.expm1(y_train_pred_log)

In [27]:
print("R²:", r2_score(np.expm1(Y_test), y_test_pred))
print("R²:", r2_score(np.expm1(Y_train), y_train_pred))

print("MAE:", mean_absolute_error(np.expm1(Y_test), y_test_pred))
print("MAE:", mean_absolute_error(np.expm1(Y_train), y_train_pred))

print("RMSE:", root_mean_squared_error(np.expm1(Y_test), y_test_pred))
print("RMSE:", root_mean_squared_error(np.expm1(Y_train), y_train_pred))

print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_test), y_test_pred))
print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_train), y_train_pred))

print("RMSLE", root_mean_squared_log_error(np.expm1(Y_test), y_test_pred))
print("RMSLE", root_mean_squared_log_error(np.expm1(Y_train), y_train_pred))

R²: 0.797771401847673
R²: 0.9281588171949325
MAE: 271983.99847546086
MAE: 172116.94643847368
RMSE: 778363.3786732241
RMSE: 491668.242318007
MAPE: 0.1442453282143657
MAPE: 0.09515555012231396
RMSLE 0.20865960842511783
RMSLE 0.13582973957089403


In [28]:
df_error = pd.DataFrame({
    "y_true": np.expm1(Y_test),
    "y_pred": y_test_pred
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_6138/101758604.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_6138/101758604.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.176408
(629900.0, 888480.0]       0.094848
(888480.0, 1300000.0]      0.122179
(1300000.0, 2000000.0]     0.135902
(2000000.0, 29998000.0]    0.191986
dtype: float64

This model gives us better results than a Random Forest or a AdaBoost for the different price ranges, but the model is overfitting.  
We can clearly observe a gab between the test set and the train set on each score (R², MAE, RMSE...)  
To reduce this overfitting, we'll reduce the depth and the max_samples for each tree. Moreover, we'll increase the min_samples_leaf and the min_samples_split.

#### Search n°2

In [27]:
gdb_search_space2 = {
    "n_estimators" : Integer(100, 800),
    "learning_rate" : Real(0.01, 0.5),

    "max_depth": Integer(2, 6),
    "min_samples_split" : Integer(30, 100),
    "min_samples_leaf" : Integer(20, 100),
    "max_features": Categorical(["sqrt", 0.60, 0.80])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

In [28]:
gdb_search2 = BayesSearchCV(
    estimator = GradientBoostingRegressor(),
    search_spaces = gdb_search_space2,
    scoring = scoring["neg_rmsle"],
    n_iter = 40,
    cv=KFold(3),
    n_jobs=7,
    error_score="raise",
    verbose=2
)

In [14]:
if os.path.isfile("../artifacts/gdb_search2.pkl"):
    print("The object already exists")
else : 
    gdb_search2.fit(X_train, Y_train)
    dump(gdb_search2, "../artifacts/gdb_search2.pkl")
    print("The object has been successfully saved")

The object already exists


In [61]:
gdb_search2 = load("../artifacts/gdb_search2.pkl")
gdb_search2.best_params_

OrderedDict([('learning_rate', 0.19703037686598415),
             ('max_depth', 6),
             ('max_features', 0.8),
             ('min_samples_leaf', 93),
             ('min_samples_split', 100),
             ('n_estimators', 800)])

In [32]:
best_model_gdb2 = gdb_search2.best_estimator_

y_pred_log2 = best_model_gdb2.predict(X_test)
y_test_pred2 = np.expm1(y_pred_log2)

y_train_pred_log2 = best_model_gdb2.predict(X_train)
y_train_pred2 = np.expm1(y_train_pred_log2)

In [33]:
print("R²:", r2_score(np.expm1(Y_test), y_test_pred2))
print("R²:", r2_score(np.expm1(Y_train), y_train_pred2))

print("MAE:", mean_absolute_error(np.expm1(Y_test), y_test_pred2))
print("MAE:", mean_absolute_error(np.expm1(Y_train), y_train_pred2))

print("RMSE:", root_mean_squared_error(np.expm1(Y_test), y_test_pred2))
print("RMSE:", root_mean_squared_error(np.expm1(Y_train), y_train_pred2))

print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_test), y_test_pred2))
print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_train), y_train_pred2))

print("RMSLE", root_mean_squared_log_error(np.expm1(Y_test), y_test_pred2))
print("RMSLE", root_mean_squared_log_error(np.expm1(Y_train), y_train_pred2))

R²: 0.8037339553350558
R²: 0.890846193219015
MAE: 276954.2110143117
MAE: 203052.64680571688
RMSE: 766802.8070103781
RMSE: 606044.9875302477
MAPE: 0.14893144707208866
MAPE: 0.10813348054180873
RMSLE 0.2128559197902954
RMSLE 0.1533050086171249


In [37]:
df_error = pd.DataFrame({
    "y_true": np.expm1(Y_test),
    "y_pred": y_test_pred2
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_500404/47697034.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_500404/47697034.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.184107
(629900.0, 888480.0]       0.100011
(888480.0, 1300000.0]      0.123040
(1300000.0, 2000000.0]     0.141172
(2000000.0, 29998000.0]    0.196437
dtype: float64

The results are slightly lower than the previous research but this new best estimator is less overfitting.  
Let's make a last search with a more restricted space.

#### Search n°3

In [47]:
gdb_search_space3 = {
    "n_estimators" : Integer(600, 700),
    "learning_rate" : Real(0.05, 0.2),

    "min_samples_split" : Integer(30, 100),
    "min_samples_leaf" : Integer(20, 100),
    "max_features": Categorical([0.80])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

In [50]:
gdb_search3 = BayesSearchCV(
    estimator = GradientBoostingRegressor(max_depth=6, random_state=42),
    search_spaces = gdb_search_space3,
    scoring = scoring["neg_rmsle"],
    n_iter = 30,
    cv=KFold(3),
    n_jobs=9,
    error_score="raise",
    verbose=2
)

In [58]:
if os.path.isfile("../artifacts/gdb_search3.pkl"):
    print("The object already exists")
else : 
    gdb_search3.fit(X_train, Y_train)
    dump(gdb_search3, "../artifacts/gdb_search3.pkl")
    print("The object has been successfully saved")

The object already exists


In [52]:
gdb_search3.best_params_

OrderedDict([('learning_rate', 0.14046032894276958),
             ('max_features', 0.8),
             ('min_samples_leaf', 30),
             ('min_samples_split', 35),
             ('n_estimators', 681)])

In [53]:
gdb_search3 = load("../artifacts/gdb_search3.pkl")
best_model_gdb3 = gdb_search3.best_estimator_

y_pred_log3 = best_model_gdb3.predict(X_test)
y_test_pred3 = np.expm1(y_pred_log3)

y_train_pred_log3 = best_model_gdb3.predict(X_train)
y_train_pred3 = np.expm1(y_train_pred_log3)

In [54]:
print("R²:", r2_score(np.expm1(Y_test), y_test_pred3))
print("R²:", r2_score(np.expm1(Y_train), y_train_pred3))

print("MAE:", mean_absolute_error(np.expm1(Y_test), y_test_pred3))
print("MAE:", mean_absolute_error(np.expm1(Y_train), y_train_pred3))

print("RMSE:", root_mean_squared_error(np.expm1(Y_test), y_test_pred3))
print("RMSE:", root_mean_squared_error(np.expm1(Y_train), y_train_pred3))

print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_test), y_test_pred3))
print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_train), y_train_pred3))

print("RMSLE", root_mean_squared_log_error(np.expm1(Y_test), y_test_pred3))
print("RMSLE", root_mean_squared_log_error(np.expm1(Y_train), y_train_pred3))

R²: 0.8026964597237225
R²: 0.9351534648052727
MAE: 275113.82850834023
MAE: 176259.77107989055
RMSE: 768826.8605316936
RMSE: 467120.3724650593
MAPE: 0.14589854803138613
MAPE: 0.0977049854349975
RMSLE 0.20984364010741488
RMSLE 0.1388874974528079


This model is overfitting a little bit more than the model from the search n°2 but it's acceptable.  
The differences between the scores on the train set and the test set are not pretty close.  

In [59]:
df_error = pd.DataFrame({
    "y_true": np.expm1(Y_test),
    "y_pred": y_test_pred3
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_500404/2607265896.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_500404/2607265896.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.177529
(629900.0, 888480.0]       0.098261
(888480.0, 1300000.0]      0.121956
(1300000.0, 2000000.0]     0.136866
(2000000.0, 29998000.0]    0.194989
dtype: float64

Furthermore, the mean average percentage error on each price range is the lowest that we have ever reached, so we'll save this model.

### We'll train our two models, from the two previous searches, on all the original dataset with the best params that we've found.  
So we'll have two trained models (One that is less overfitting and the second one that has more variance in its predictions  
than the fist one and better results on the MAPE on each price ranges).  
Moreover, We'll use the MAPE metrics for each price ranges in order to provide an interval for each estimation provided by our final model.  
The user will receive a mean price and a min/max price around it.

In [9]:
gdb_final_model1 = GradientBoostingRegressor(
    max_depth = 6,
    learning_rate =  0.197,
    max_features = 0.8,
    min_samples_leaf = 93,
    min_samples_split = 100,
    n_estimators = 800,
    random_state = 42,
    verbose = 1
)

gdb_final_model2 = GradientBoostingRegressor(
    max_depth = 6,
    learning_rate = 0.14,
    max_features = 0.8,
    min_samples_leaf = 30,
    min_samples_split = 35,
    n_estimators = 681,
    random_state = 42,
    verbose = 1
)

In [12]:
if os.path.isfile("../artifacts/gdb_final_model1.pkl"):
    print("The object already exists")
else : 
    gdb_final_model1.fit(X, Y_log)
    dump(gdb_final_model1, "../artifacts/gdb_final_model1.pkl")
    print("The object has been successfully saved")

The object already exists


In [13]:
if os.path.isfile("../artifacts/gdb_final_model2.pkl"):
    print("The object already exists")
else : 
    gdb_final_model2.fit(X, Y_log)
    dump(gdb_final_model2, "../artifacts/gdb_final_model2.pkl")
    print("The object has been successfully saved")

The object already exists


We create and save the dictionnary of the mape metrics of each price range

In [6]:
if os.path.isfile("../artifacts/gdb_mape_dict.pkl"):
    print("The object already exists")
else :
    # If the price estimate is lower or equal to 629'900$, we'll create a price range that coressponds to ± "price prediction".
    gdb_mape_dict = {
        (49499.999, 629900.0): 0.178,
        (629900.0, 888480.0): 0.098,
        (888480.0, 1300000.0): 0.12,
        (1300000.0, 2000000.0): 0.137,
        (2000000.0, 29998000.0): 0.19
        # If the price estimate exceeds 2'000'000$, the range will be equal to ± "price prediction".
    }
    dump(gdb_mape_dict, "../artifacts/gdb_mape_dict.pkl")
    print("The object has been successfully saved")

TypeError: unhashable type: 'list'